<div class="blog-language-switch" role="group" aria-label="Article language">
<span aria-current="page">English</span>
<a href="../zh-CN/Deep-Learning/10-graph-neural-networks-geometric-deep-learning.html" lang="zh-CN" hreflang="zh-CN">中文</a>
</div>

[Back to Deep Learning guideline](Deep-Learning.html)

## **Graph Neural Networks and Geometric Deep Learning** {#graph-neural-networks-geometric-deep-learning}

Images, audio, and ordinary sequences live on regular grids: every pixel or token has a predictable coordinate and a fixed notion of the next position. A graph instead represents a set of entities and a relation set $G=(V,E)$. The number of neighbors varies by node, node identifiers are arbitrary, and changing the order in which nodes are stored must not change the represented object. **Geometric deep learning** studies architectures whose computation respects such symmetries and domains; graph neural networks (GNNs) are its most widely used discrete form.

This chapter uses one real network from beginning to end: [Zachary's Karate Club graph in NetworkX](https://networkx.org/documentation/stable/reference/generated/networkx.generators.social.karate_club_graph.html). Wayne Zachary recorded interactions among 34 members and later observed the club split around two leaders. NetworkX exposes 78 undirected edges, an interaction-count edge weight, and the observed faction as the node attribute `club`. The NetworkX implementation is distributed under BSD-3-Clause; its documentation cites the [original 1977 study](https://www.jstor.org/stable/3629752) and does not state a separate data license. The chapter therefore uses only the small topology and metadata shipped by NetworkX, with explicit attribution.

The fixed experiment is **transductive node classification**: eight labeled nodes train the model, eight select a checkpoint, and eighteen are held out for a final mechanism check, while the whole graph topology remains visible. Later, a clean edge split supports link prediction. This network is too small, historically specific, and socially correlated to establish general GNN superiority. Its value is that every matrix, neighborhood, attention coefficient, and leakage boundary can be inspected.

![The Zachary Karate Club network, colored by the two factions observed after the split.](assets/dl10-karate-club.svg){fig-align="center" width="76%" fig-alt="Zachary Karate Club graph with 34 numbered nodes, two faction colors, and weighted interaction edges."}

*Original data visualization generated for this chapter from the [NetworkX Karate Club graph](https://networkx.org/documentation/stable/reference/generated/networkx.generators.social.karate_club_graph.html); topology and metadata trace to Zachary (1977).*

### **Why Graph-Structured Data Is Different** {#why-graph-structured-data-is-different}

A graph is useful when relationships are part of the input rather than an accidental correlation. In a molecule, atoms are nodes and bonds are edges; in recommendation, users and items form a bipartite interaction graph; in a road network, intersections connect through directed, weighted roads. Flattening these objects into a vector discards which features belong to adjacent entities. Placing them on an arbitrary dense grid invents spatial neighbors that do not exist.

Three properties determine the model design. First, **irregular neighborhoods** require aggregation over a variable-size set. Second, **permutation symmetry** requires node outputs to permute with node ordering and graph outputs to remain unchanged. Third, **relational dependence** means examples are not automatically independent: random node or edge splits can leak labels, future events, duplicate entities, or held-out edges through message passing.

GNNs address the first two properties by sharing a local update rule across nodes and using order-invariant aggregation. They do not automatically solve the third. Evaluation protocol, feature provenance, graph construction, and deployment assumptions are as important as the layer equation.

<details>
<summary><strong>PyTorch and NetworkX: establish the shared Karate Club experiment</strong></summary>

```python
import copy
import random

import networkx as nx
import numpy as np
import torch
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, f1_score, roc_auc_score
from torch import nn
from torch.nn import functional as F

torch.set_num_threads(1)


def seed_everything(seed=1010):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


seed_everything()
graph = nx.karate_club_graph()
nodes = sorted(graph.nodes())
num_nodes = graph.number_of_nodes()

# Binary topology drives message passing; the original interaction counts are
# retained separately as edge attributes for a later relation-aware example.
adjacency = torch.tensor(
    nx.to_numpy_array(graph, nodelist=nodes, weight=None), dtype=torch.float32
)
weighted_adjacency = torch.tensor(
    nx.to_numpy_array(graph, nodelist=nodes, weight="weight"), dtype=torch.float32
)
labels = torch.tensor(
    [0 if graph.nodes[node]["club"] == "Mr. Hi" else 1 for node in nodes],
    dtype=torch.long,
)

undirected_edges = torch.tensor(list(graph.edges()), dtype=torch.long).T
edge_index = torch.cat([undirected_edges, undirected_edges.flip(0)], dim=1)
degree = adjacency.sum(1)
clustering = torch.tensor(
    [nx.clustering(graph, node) for node in nodes], dtype=torch.float32
)
node_features = torch.stack(
    [degree / degree.max(), clustering, torch.ones(num_nodes)], dim=1
)
identity_features = torch.eye(num_nodes)

# One deterministic, class-stratified transductive split is reused throughout.
rng = np.random.default_rng(1010)
train_nodes, val_nodes, test_nodes = [], [], []
for class_id in (0, 1):
    members = np.flatnonzero(labels.numpy() == class_id)
    rng.shuffle(members)
    train_nodes.extend(members[:4])
    val_nodes.extend(members[4:8])
    test_nodes.extend(members[8:])


def mask_from(indices):
    mask = torch.zeros(num_nodes, dtype=torch.bool)
    mask[torch.tensor(indices, dtype=torch.long)] = True
    return mask


train_mask = mask_from(train_nodes)
val_mask = mask_from(val_nodes)
test_mask = mask_from(test_nodes)

assert num_nodes == 34 and graph.number_of_edges() == 78
assert torch.equal(adjacency, adjacency.T) and torch.all(adjacency.diag() == 0)
assert train_mask.sum() == 8 and val_mask.sum() == 8 and test_mask.sum() == 18
assert not (train_mask & val_mask).any() and not (train_mask & test_mask).any()
print({
    "NetworkX": nx.__version__,
    "nodes": num_nodes,
    "undirected edges": graph.number_of_edges(),
    "split": (int(train_mask.sum()), int(val_mask.sum()), int(test_mask.sum())),
    "feature shape": tuple(node_features.shape),
})
```

</details>

The three structural features are normalized degree, local clustering coefficient, and a constant channel. Identity features are also retained because classic semi-supervised GCN demonstrations often learn a separate basis direction for every node. That representation is explicitly transductive: a new node has no trained identity column. Later GraphSAGE examples use structural features to expose the difference between learning node IDs and learning a reusable neighborhood function.


### **Graph Representations and Learning Tasks** {#graph-representations-learning-tasks}

The same graph can be stored as an edge list, adjacency matrix, sparse coordinate list, or neighborhood dictionary. For $N=|V|$ nodes, a dense adjacency matrix $A\in\{0,1\}^{N\times N}$ has $A_{ij}=1$ when an edge connects $i$ and $j$. It costs $O(N^2)$ memory even if the graph has only $E\ll N^2$ edges. An edge index stores endpoint pairs and costs $O(E)$; an undirected edge is commonly expanded into two directed messages, so this chapter's 78 relations become 156 columns.

Node features form $X\in\mathbb{R}^{N\times F}$, edge features form $E_f\in\mathbb{R}^{|E|\times F_e}$, and optional graph-level features describe the whole object. A relabeling represented by permutation matrix $P$ produces $X'=PX$ and $A'=PAP^\top$. A node model should satisfy

$$
f(PX,PAP^\top)=P f(X,A),
$$

which is **permutation equivariance**. A graph readout $r$ should instead satisfy $r(PX,PAP^\top)=r(X,A)$, which is **permutation invariance**. These are architectural contracts, not data-augmentation preferences.

The prediction target determines the output structure. **Node-level** tasks classify members, papers, or atoms. **Edge-level** tasks predict links, relation types, or bond properties. **Graph-level** tasks map an entire molecule, program graph, or scene to one label or scalar. A fourth family performs **graph generation**, where validity constraints and permutation-equivalent likelihoods require additional machinery.

![Node, edge, and graph targets require different readout operations.](assets/dl10-readout-levels.svg){fig-align="center" width="72%" fig-alt="Three panels comparing node classification, edge scoring, and graph-level invariant pooling."}

*Original teaching diagram based on the task taxonomy in [Distill's GNN introduction](https://distill.pub/2021/gnn-intro/), whose text and diagrams are CC BY 4.0.*

<details>
<summary><strong>Python: compare edge-list, adjacency, and sparse edge-index views</strong></summary>

```python
edge_list = sorted(tuple(sorted(edge)) for edge in graph.edges())
neighbors_of_zero = sorted(graph.neighbors(0))

# Relabeling a graph is a simultaneous permutation of rows and columns.
permutation = torch.randperm(num_nodes, generator=torch.Generator().manual_seed(1011))
permuted_adjacency = adjacency[permutation][:, permutation]
permuted_features = node_features[permutation]

assert len(edge_list) == 78
assert edge_index.shape == (2, 156)  # both directions are explicit
assert torch.equal(permuted_adjacency.sum(1), degree[permutation])
assert torch.equal(permuted_features[:, -1], torch.ones(num_nodes))
print({
    "node 0 neighbors": neighbors_of_zero,
    "adjacency shape": tuple(adjacency.shape),
    "edge-index shape": tuple(edge_index.shape),
    "density": float(adjacency.sum() / num_nodes**2),
})
```

</details>

Dense matrices are ideal for a 34-node teaching graph because the algebra remains visible. Production libraries such as PyTorch Geometric and DGL use sparse message-passing kernels so a layer is closer to $O(EF)$ than $O(N^2F)$. Sparse storage does not remove high-degree load imbalance, duplicate-edge semantics, or expensive neighborhood expansion; those remain systems concerns.


### **The Message-Passing Framework** {#message-passing-framework}

Most practical GNN layers can be decomposed into **message**, **aggregate**, and **update** operations. For node $i$ at layer $k$,

$$
m_{ji}^{(k)}=\phi^{(k)}\!\left(h_i^{(k-1)},h_j^{(k-1)},e_{ji}\right),\qquad
m_i^{(k)}=\bigoplus_{j\in\mathcal{N}(i)}m_{ji}^{(k)},\qquad
h_i^{(k)}=\gamma^{(k)}\!\left(h_i^{(k-1)},m_i^{(k)}\right).
$$

$h_i^{(k)}$ is node $i$'s representation, $e_{ji}$ is an optional edge feature, $\phi$ constructs a directed message, $\bigoplus$ is an order-invariant operator such as sum, mean, or max, and $\gamma$ combines the aggregated neighborhood with the previous state. This is the same abstraction formalized by the [PyTorch Geometric `MessagePassing` API](https://pytorch-geometric.readthedocs.io/en/latest/tutorial/create_gnn.html).

![A message-passing layer applies a shared message function, invariant aggregation, and a node update.](assets/dl10-message-passing.svg){fig-align="center" width="74%" fig-alt="Message passing process from three neighboring nodes through message, aggregate, and update stages."}

*Original teaching diagram created from the message-passing equation in the [PyTorch Geometric documentation](https://pytorch-geometric.readthedocs.io/en/latest/tutorial/create_gnn.html) and the [Distill GNN introduction](https://distill.pub/2021/gnn-intro/).*

After one layer, $h_i$ depends on one-hop neighbors; after $K$ layers, it can depend on nodes within $K$ hops. This **receptive field** statement describes possible information paths, not guaranteed usable information. Aggregation may lose multiplicity, normalization may attenuate signals, and narrow cuts may compress distant evidence.

<details>
<summary><strong>PyTorch: implement one permutation-equivariant message-passing layer</strong></summary>

```python
def mean_aggregate(features, binary_adjacency, include_self=True):
    mixing = binary_adjacency.clone()
    if include_self:
        mixing = mixing + torch.eye(mixing.shape[0])
    return mixing @ features / mixing.sum(1, keepdim=True).clamp_min(1.0)


class MessagePassingLayer(nn.Module):
    def __init__(self, input_dim, output_dim):
        super().__init__()
        self.update = nn.Linear(2 * input_dim, output_dim)

    def forward(self, features, binary_adjacency):
        neighbor_message = mean_aggregate(features, binary_adjacency)
        return torch.relu(self.update(torch.cat([features, neighbor_message], dim=1)))


seed_everything(1012)
message_layer = MessagePassingLayer(node_features.shape[1], 8)
node_states = message_layer(node_features, adjacency)
permuted_states = message_layer(permuted_features, permuted_adjacency)

# The output follows the node permutation: graph labels may change, semantics do not.
assert node_states.shape == (num_nodes, 8)
assert torch.allclose(permuted_states, node_states[permutation], atol=1e-6)
print({"node 0 state": node_states[0].detach().round(decimals=3).tolist(),
       "permutation equivariant": True})
```

</details>

The example uses a mean and concatenates the target's own state before a learned update. Mean aggregation controls scale across degrees but cannot distinguish neighborhoods with equal averages. Sum preserves multiplicity and is more expressive for counting, yet its magnitude grows with neighborhood size. Max highlights dominant coordinates but discards frequency. The aggregation choice therefore encodes an inductive bias rather than a universally best default.


### **Graph Convolutional Networks** {#graph-convolutional-networks}

The Graph Convolutional Network (GCN) of [Kipf and Welling](https://arxiv.org/abs/1609.02907) uses a fixed, degree-normalized neighborhood average followed by a learned channel transformation. Add self-loops $\hat A=A+I$, define $\hat D_{ii}=\sum_j\hat A_{ij}$, and propagate

$$
H^{(k+1)}=\sigma\!\left(\hat D^{-1/2}\hat A\hat D^{-1/2}H^{(k)}W^{(k)}\right).
$$

$H^{(k)}\in\mathbb{R}^{N\times F_k}$ contains node states, $W^{(k)}\in\mathbb{R}^{F_k\times F_{k+1}}$ mixes channels, and the symmetric factors divide each edge contribution by $\sqrt{\hat d_i\hat d_j}$. High-degree nodes therefore neither accumulate an uncontrolled sum nor dominate every low-degree neighbor. The published layer arose from a localized first-order approximation to spectral graph convolution, but it is also understandable directly as normalized message passing.

Two layers let every node use two-hop evidence. In the transductive Karate experiment, the loss is evaluated only on `train_mask`, while unlabeled nodes still participate in propagation. This is valid only because deployment assumes the same graph is available. If test nodes or their edges should be unseen at training time, this protocol is wrong even though the labels are masked.

![GCN, GraphSAGE, and GAT differ primarily in how they choose and weight neighborhood messages.](assets/dl10-gnn-operators.svg){fig-align="center" width="78%" fig-alt="Three-panel comparison of degree-normalized GCN, sampled mean GraphSAGE, and attention-weighted GAT aggregation."}

*Original comparison diagram based on the [GCN](https://arxiv.org/abs/1609.02907), [GraphSAGE](https://arxiv.org/abs/1706.02216), and [GAT](https://arxiv.org/abs/1710.10903) papers.*

<details>
<summary><strong>PyTorch: train a two-layer GCN from matrix operations</strong></summary>

```python
def gcn_normalize(binary_adjacency):
    adjacency_with_self = binary_adjacency + torch.eye(binary_adjacency.shape[0])
    inverse_sqrt_degree = adjacency_with_self.sum(1).pow(-0.5)
    return (
        inverse_sqrt_degree[:, None]
        * adjacency_with_self
        * inverse_sqrt_degree[None, :]
    )


class KarateGCN(nn.Module):
    def __init__(self, propagation, input_dim, hidden_dim=16):
        super().__init__()
        self.propagation = propagation
        self.input_layer = nn.Linear(input_dim, hidden_dim, bias=False)
        self.output_layer = nn.Linear(hidden_dim, 2, bias=False)

    def forward(self, features):
        hidden = torch.relu(self.propagation @ self.input_layer(features))
        logits = self.propagation @ self.output_layer(hidden)
        return logits, hidden


def fit_node_model(model, features, epochs=300, learning_rate=0.02):
    optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=5e-4)
    best_val_loss = float("inf")
    best_state = None
    for _ in range(epochs):
        model.train()
        logits, _ = model(features)
        loss = F.cross_entropy(logits[train_mask], labels[train_mask])
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        model.eval()
        with torch.no_grad():
            val_logits, _ = model(features)
            val_loss = F.cross_entropy(val_logits[val_mask], labels[val_mask]).item()
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = copy.deepcopy(model.state_dict())
    model.load_state_dict(best_state)
    model.eval()
    with torch.no_grad():
        logits, hidden = model(features)
    predictions = logits.argmax(1)
    return {
        "validation loss": best_val_loss,
        "test macro F1": f1_score(labels[test_mask], predictions[test_mask], average="macro"),
    }, logits, hidden


seed_everything(1013)
normalized_adjacency = gcn_normalize(adjacency)
gcn_model = KarateGCN(normalized_adjacency, input_dim=num_nodes)
gcn_metrics, gcn_logits, gcn_embeddings = fit_node_model(gcn_model, identity_features)

assert torch.allclose(normalized_adjacency, normalized_adjacency.T)
assert gcn_logits.shape == (num_nodes, 2) and gcn_embeddings.shape == (num_nodes, 16)
print({key: round(float(value), 4) for key, value in gcn_metrics.items()})
```

</details>

The identity input gives every observed member a trainable basis direction and makes the topology itself sufficient for a demonstration. It cannot embed member 34 without changing the input layer. The reported macro F1 comes from one tiny, fixed split and should be read alongside validation loss and simple baselines, not as evidence that GCN is generally best. On large sparse graphs, the dense multiplications shown here must become sparse gather/scatter operations.


### **GraphSAGE** {#graphsage}

[GraphSAGE](https://arxiv.org/abs/1706.02216) changes the question from “what embedding parameter belongs to node $i$?” to “what function maps a node's features and sampled neighborhood to an embedding?” A mean-aggregator layer can be written

$$
\bar h_{\mathcal N(i)}^{(k)}=\frac{1}{|S_k(i)|}\sum_{j\in S_k(i)}h_j^{(k-1)},\qquad
h_i^{(k)}=\sigma\!\left(W^{(k)}[h_i^{(k-1)}\Vert \bar h_{\mathcal N(i)}^{(k)}]\right),
$$

where $S_k(i)\subseteq\mathcal N(i)$ is a sampled neighbor set and $\Vert$ denotes concatenation. Because parameters belong to the aggregation function rather than a node lookup table, the same function can process a new node if its features and neighborhood are available. This is an **inductive interface**, not an automatic guarantee that distribution-shifted nodes will be accurate.

Sampling limits a $K$-layer computation to approximately $\prod_k s_k$ sampled paths per target instead of expanding every neighbor. It reduces cost and enables mini-batches, but introduces estimator variance. High-degree neighborhoods may be underrepresented, rare relation types can disappear, and naive recursive sampling repeats work for overlapping receptive fields.

<details>
<summary><strong>PyTorch: train a mean GraphSAGE layer and inspect sampling error</strong></summary>

```python
class MeanSAGELayer(nn.Module):
    def __init__(self, input_dim, output_dim):
        super().__init__()
        self.linear = nn.Linear(2 * input_dim, output_dim)

    def forward(self, features, binary_adjacency):
        neighbor_mean = mean_aggregate(features, binary_adjacency, include_self=False)
        return torch.relu(self.linear(torch.cat([features, neighbor_mean], dim=1)))


class KarateGraphSAGE(nn.Module):
    def __init__(self, binary_adjacency, input_dim, hidden_dim=16):
        super().__init__()
        self.binary_adjacency = binary_adjacency
        self.sage = MeanSAGELayer(input_dim, hidden_dim)
        self.classifier = nn.Linear(hidden_dim, 2)

    def forward(self, features):
        hidden = self.sage(features, self.binary_adjacency)
        return self.classifier(hidden), hidden


def sampled_adjacency(graph_object, fan_out, seed):
    generator = random.Random(seed)
    sampled = torch.zeros((num_nodes, num_nodes))
    for target in nodes:
        candidates = list(graph_object.neighbors(target))
        chosen = generator.sample(candidates, min(fan_out, len(candidates)))
        sampled[target, chosen] = 1.0
    return sampled


seed_everything(1014)
sage_model = KarateGraphSAGE(adjacency, input_dim=node_features.shape[1])
sage_metrics, sage_logits, sage_embeddings = fit_node_model(
    sage_model, node_features, learning_rate=0.015
)
sampled_graph = sampled_adjacency(graph, fan_out=2, seed=1014)
full_neighbor_mean = mean_aggregate(node_features, adjacency, include_self=False)
sampled_neighbor_mean = mean_aggregate(node_features, sampled_graph, include_self=False)

assert sampled_graph.sum(1).max() <= 2
assert sage_embeddings.shape == (num_nodes, 16)
print({
    "sampled directed messages": int(sampled_graph.sum()),
    "node 0 mean approximation error": round(
        float((full_neighbor_mean[0] - sampled_neighbor_mean[0]).norm()), 4
    ),
    "test macro F1": round(float(sage_metrics["test macro F1"]), 4),
})
```

</details>

This model receives degree, clustering, and a constant rather than node identities. Those features can be computed for another graph, although their ranges and meanings may shift. The two-neighbor sampler deliberately creates visible approximation error; real systems choose fan-outs per layer, sample by relation or importance, cache neighborhoods, and report variance across sampling seeds.


### **Graph Attention Networks** {#graph-attention-networks}

GCN fixes neighbor weights from degrees. A [Graph Attention Network (GAT)](https://arxiv.org/abs/1710.10903) learns them from node content. For transformed states $z_i=Wh_i$, the original single-head layer computes

$$
e_{ij}=\operatorname{LeakyReLU}\!\left(a^\top[z_i\Vert z_j]\right),\qquad
\alpha_{ij}=\frac{\exp(e_{ij})}{\sum_{k\in\mathcal N(i)\cup\{i\}}\exp(e_{ik})},\qquad
h_i'=\sigma\!\left(\sum_j\alpha_{ij}z_j\right).
$$

The softmax is local: coefficients sum to one **within each target node's incoming neighborhood**, not over the whole graph. Multi-head GAT repeats the calculation with separate projections. Intermediate heads are often concatenated to increase capacity; final heads may be averaged to stabilize predictions.

Attention can suppress unhelpful neighbors and adapt weights across examples, but it is not free. It computes edge-wise scores, stores coefficients, and still exchanges information only along supplied edges. The original additive scoring form also has representational limitations; learned coefficients should not be treated as causal explanations without intervention.

<details>
<summary><strong>PyTorch: implement local edge softmax and train a single-head GAT</strong></summary>

```python
class SingleHeadGATLayer(nn.Module):
    def __init__(self, input_dim, output_dim):
        super().__init__()
        self.projection = nn.Linear(input_dim, output_dim, bias=False)
        self.attention_source = nn.Parameter(torch.empty(output_dim))
        self.attention_target = nn.Parameter(torch.empty(output_dim))
        nn.init.xavier_uniform_(self.projection.weight)
        nn.init.normal_(self.attention_source, std=0.2)
        nn.init.normal_(self.attention_target, std=0.2)

    def forward(self, features, directed_edges):
        self_nodes = torch.arange(features.shape[0])
        edges = torch.cat([directed_edges, torch.stack([self_nodes, self_nodes])], dim=1)
        source, target = edges
        transformed = self.projection(features)
        scores = F.leaky_relu(
            (transformed[source] * self.attention_source).sum(1)
            + (transformed[target] * self.attention_target).sum(1),
            negative_slope=0.2,
        )
        coefficients = torch.zeros_like(scores)
        for target_node in range(features.shape[0]):
            incoming = target == target_node
            coefficients[incoming] = torch.softmax(scores[incoming], dim=0)
        output = torch.zeros_like(transformed)
        output.index_add_(0, target, coefficients[:, None] * transformed[source])
        return F.elu(output), edges, coefficients


class KarateGAT(nn.Module):
    def __init__(self, input_dim, hidden_dim=16):
        super().__init__()
        self.gat = SingleHeadGATLayer(input_dim, hidden_dim)
        self.classifier = nn.Linear(hidden_dim, 2)
        self.last_edges = None
        self.last_attention = None

    def forward(self, features):
        hidden, edges, coefficients = self.gat(features, edge_index)
        self.last_edges = edges
        self.last_attention = coefficients.detach()
        return self.classifier(hidden), hidden


seed_everything(1015)
gat_model = KarateGAT(node_features.shape[1])
gat_metrics, gat_logits, gat_embeddings = fit_node_model(
    gat_model, node_features, learning_rate=0.012
)
with torch.no_grad():
    gat_model(node_features)
target_indices = gat_model.last_edges[1]
attention_sums = torch.zeros(num_nodes)
attention_sums.index_add_(0, target_indices, gat_model.last_attention)

assert torch.allclose(attention_sums, torch.ones(num_nodes), atol=1e-5)
assert gat_embeddings.shape == (num_nodes, 16)
print({"node 0 incoming coefficients": gat_model.last_attention[target_indices == 0].round(decimals=3).tolist(),
       "test macro F1": round(float(gat_metrics["test macro F1"]), 4)})
```

</details>

The explicit loop is appropriate only for 34 nodes because it makes the normalization domain undeniable. Sparse libraries implement a segment softmax over target indices. A common bug is normalizing all edges globally, which couples unrelated neighborhoods; another is swapping source and target indices and silently normalizing outgoing rather than incoming messages.


### **Node, Edge, and Graph-Level Readout** {#node-edge-graph-readout}

Message passing creates node states; the task head decides how those states become predictions. A node classifier applies an MLP or linear map to each $h_i$. An edge predictor combines endpoint states through a dot product, distance, bilinear form, Hadamard product, or relation-aware MLP. A graph predictor pools all node or edge states into one representation.

For graph-level output, a readout must be permutation invariant:

$$
h_G=\operatorname{READOUT}\left(\{h_i:i\in V\}\right).
$$

`sum` preserves graph-size information and can count repeated structures; `mean` compares average composition across sizes; `max` detects whether a strong feature exists. Attention pooling and virtual global nodes are learnable alternatives, but they still need an invariant treatment of the input set. Hierarchical pooling coarsens the graph when local substructures should become higher-level units.

<details>
<summary><strong>PyTorch: derive node, edge, and graph outputs from one embedding table</strong></summary>

```python
with torch.no_grad():
    node_logits, node_embeddings = gcn_model(identity_features)

unique_source, unique_target = undirected_edges
edge_embeddings = node_embeddings[unique_source] * node_embeddings[unique_target]
edge_scores = edge_embeddings.sum(1)
graph_mean = node_embeddings.mean(0)
graph_sum = node_embeddings.sum(0)
graph_max = node_embeddings.max(0).values

permuted_embeddings = node_embeddings[permutation]
assert node_logits.shape == (34, 2)
assert edge_scores.shape == (78,)
assert torch.allclose(graph_mean, permuted_embeddings.mean(0), atol=1e-6)
assert torch.allclose(graph_sum, graph_mean * num_nodes, atol=1e-6)
print({"node output": tuple(node_logits.shape), "edge output": tuple(edge_scores.shape),
       "graph readouts": [tuple(x.shape) for x in (graph_mean, graph_sum, graph_max)]})
```

</details>

The Hadamard edge representation is symmetric, so swapping endpoints leaves an undirected score unchanged. Concatenation is order sensitive unless both orders are modeled. Likewise, `sum` and `mean` are not interchangeable: the equation `sum = mean × number of nodes` shows exactly which size signal mean removes.


### **Heterogeneous and Temporal Graphs** {#heterogeneous-temporal-graphs}

A homogeneous graph assumes one node type and one edge semantics. Knowledge graphs, recommender systems, and software dependency graphs violate that assumption. A heterogeneous message-passing layer conditions on relation type $r$:

$$
h_i'=\gamma\!\left(h_i,\sum_r\sum_{j\in\mathcal N_r(i)}\phi_r(h_i,h_j,e_{ji})\right).
$$

Relation-specific matrices are simple but scale linearly with the number of relations. Basis decomposition, parameter sharing, typed attention, and metapath methods trade capacity for statistical and memory efficiency. Node types may also have different raw feature spaces, requiring type-specific encoders before messages share a hidden dimension.

A temporal graph adds event time: $(u,v,t,e_{uvt})$. The neighborhood available for a prediction at time $t$ must contain only events with $t'<t$. Temporal GNNs may maintain node memories, encode time gaps, or sample recent historical neighbors. A random edge split is invalid when deployment predicts the future because it allows future interactions to shape past embeddings.

<details>
<summary><strong>PyTorch: apply relation-specific messages without inventing timestamps</strong></summary>

```python
# The original edge weight counts interaction contexts. We bin it only to
# demonstrate relation-specific parameters; this is not a native edge schema.
relation_edges = {"lower_weight": [], "higher_weight": []}
for source, target, attributes in graph.edges(data=True):
    relation = "higher_weight" if attributes["weight"] >= 3 else "lower_weight"
    relation_edges[relation].extend([(source, target), (target, source)])

seed_everything(1016)
relation_weights = {
    "lower_weight": torch.randn(3, 4) / 3**0.5,
    "higher_weight": torch.randn(3, 4) / 3**0.5,
}
relation_output = torch.zeros(num_nodes, 4)
for relation, edges in relation_edges.items():
    directed = torch.tensor(edges, dtype=torch.long).T
    source, target = directed
    messages = node_features[source] @ relation_weights[relation]
    relation_output.index_add_(0, target, messages)

has_timestamps = any("timestamp" in attributes for *_, attributes in graph.edges(data=True))
assert relation_output.shape == (34, 4)
assert sum(len(edges) for edges in relation_edges.values()) == 156
assert not has_timestamps
print({key: len(value) // 2 for key, value in relation_edges.items()},
      "timestamps available:", has_timestamps)
```

</details>

Karate Club supplies interaction counts but no event timestamps. The code bins weights into two **constructed** relations to demonstrate parameter routing; it does not claim that Zachary recorded two edge types. More importantly, it refuses to simulate a temporal conclusion from weights. Interaction strength answers “how often across observed contexts,” while time answers “when”; substituting one for the other would create false causality.


### **Oversmoothing and Oversquashing** {#oversmoothing-oversquashing}

Deep message passing has two distinct failure modes. **Oversmoothing** occurs when repeated neighborhood mixing makes node representations increasingly similar. GCN propagation behaves like Laplacian smoothing; after enough layers on a connected graph, discriminative variation can collapse toward a low-dimensional stationary subspace. [Li, Han, and Wu](https://arxiv.org/abs/1801.07606) identify this smoothing as both a source of GCN effectiveness and a depth limitation.

**Oversquashing** occurs when a rapidly expanding receptive field must compress many distant signals through fixed-width states or a narrow graph cut. [Alon and Yahav](https://arxiv.org/abs/2006.05205) show that long-range tasks can fail even before every node representation becomes similar. Oversmoothing is about states becoming indistinguishable; oversquashing is about relevant remote information failing to traverse a bottleneck.

![Oversmoothing collapses distinctions, whereas oversquashing compresses many long-range messages through a narrow path.](assets/dl10-failure-modes.svg){fig-align="center" width="76%" fig-alt="Two-panel diagram contrasting representation collapse from oversmoothing with a narrow information bottleneck causing oversquashing."}

*Original teaching diagram based on [Deeper Insights into GCNs](https://arxiv.org/abs/1801.07606) and [On the Bottleneck of GNNs](https://arxiv.org/abs/2006.05205).*

<details>
<summary><strong>Python: measure smoothing and inspect topology bottleneck proxies</strong></summary>

```python
# A row-stochastic propagation matrix isolates the smoothing effect.
random_walk = adjacency + torch.eye(num_nodes)
random_walk = random_walk / random_walk.sum(1, keepdim=True)
states = node_features.clone()
diagnostics = []
for layer in range(21):
    class_gap = (states[labels == 0].mean(0) - states[labels == 1].mean(0)).norm()
    diagnostics.append((layer, float(states.var(0).mean()), float(class_gap)))
    states = random_walk @ states

# Topology diagnostics for possible information bottlenecks.
eccentricity = nx.eccentricity(graph)
peripheral_target = max(eccentricity, key=eccentricity.get)
distances = nx.single_source_shortest_path_length(graph, peripheral_target)
shell_sizes = {
    distance: sum(value == distance for value in distances.values())
    for distance in sorted(set(distances.values()))
}
edge_betweenness = nx.edge_betweenness_centrality(graph)
highest_betweenness_edge = max(edge_betweenness, key=edge_betweenness.get)

assert diagnostics[-1][1] < diagnostics[0][1]
assert sum(shell_sizes.values()) == num_nodes
print({
    "variance layer 0 -> 20": (round(diagnostics[0][1], 5), round(diagnostics[-1][1], 5)),
    "class gap layer 0 -> 20": (round(diagnostics[0][2], 4), round(diagnostics[-1][2], 4)),
    "peripheral target and shells": (peripheral_target, shell_sizes),
    "highest edge-betweenness candidate": highest_betweenness_edge,
})
```

</details>

Decreasing node-feature variance is a smoothing diagnostic, not a complete proof of task failure. Edge betweenness and receptive-field shell size are likewise topology proxies, not direct measurements of information capacity. Useful remedies include residual or initial-feature connections, normalization, shallow models, jumping knowledge, graph rewiring, positional/structural encodings, and global attention. Each remedy changes a different cause, so diagnosis should precede architecture changes.


### **Molecular Modeling and Recommendation Systems** {#molecular-modeling-recommendation-systems}

GNN abstractions transfer across domains only after the graph semantics are specified. In molecular learning, atoms carry element, charge, aromaticity, and geometry; bonds carry type and sometimes distance. Node tasks predict atom properties, edge tasks predict bonds or interactions, and graph readouts predict molecular properties. Three-dimensional models additionally require rotation/translation invariance or equivariance; a plain 2D GCN does not acquire physical symmetry merely because the input is called a molecule.

In recommendation, users and items are different node types and observed interactions are edges. The model scores a candidate pair, often with a dot product or MLP. Unobserved pairs are not necessarily negative: the user may never have seen the item. Negative sampling therefore defines the learning problem, and popularity-biased exposure can make offline ranking metrics optimistic.

The Karate graph can support an edge-prediction mechanism check. To avoid structural leakage, validation and test positive edges are removed from the adjacency used by the encoder. The removal algorithm keeps the training graph connected, then samples true non-edges as negatives. Identity features make this a transductive link model, analogous to learned user/item IDs; it cannot score a brand-new entity without side features or an inductive encoder.

<details>
<summary><strong>PyTorch: train a leakage-aware link predictor on held-out Karate edges</strong></summary>

```python
def make_connected_edge_split(graph_object, validation_size=6, test_size=6, seed=1017):
    generator = random.Random(seed)
    candidates = list(graph_object.edges())
    generator.shuffle(candidates)
    training_graph = graph_object.copy()
    held_out = []
    for edge in candidates:
        training_graph.remove_edge(*edge)
        if nx.is_connected(training_graph):
            held_out.append(edge)
        else:
            training_graph.add_edge(*edge, **graph_object.edges[edge])
        if len(held_out) == validation_size + test_size:
            break
    return training_graph, held_out[:validation_size], held_out[validation_size:]


def edge_tensor(edges):
    return torch.tensor(edges, dtype=torch.long).T.contiguous()


def dot_scores(embeddings, edges):
    source, target = edges
    return (embeddings[source] * embeddings[target]).sum(1)


training_graph, validation_positive, test_positive = make_connected_edge_split(graph)
training_positive = list(training_graph.edges())
negative_edges = list(nx.non_edges(graph))
random.Random(1017).shuffle(negative_edges)
training_negative = negative_edges[:len(training_positive)]
validation_negative = negative_edges[len(training_positive):len(training_positive) + len(validation_positive)]
test_negative = negative_edges[len(training_positive) + len(validation_positive):
                               len(training_positive) + len(validation_positive) + len(test_positive)]

training_adjacency = torch.tensor(
    nx.to_numpy_array(training_graph, nodelist=nodes, weight=None), dtype=torch.float32
)


class LinkEncoder(nn.Module):
    def __init__(self, propagation, hidden_dim=16):
        super().__init__()
        self.propagation = propagation
        self.layer1 = nn.Linear(num_nodes, hidden_dim, bias=False)
        self.layer2 = nn.Linear(hidden_dim, hidden_dim, bias=False)

    def forward(self, features):
        hidden = torch.relu(self.propagation @ self.layer1(features))
        return self.propagation @ self.layer2(hidden)


def binary_edge_loss(embeddings, positive, negative):
    positive_scores = dot_scores(embeddings, edge_tensor(positive))
    negative_scores = dot_scores(embeddings, edge_tensor(negative))
    scores = torch.cat([positive_scores, negative_scores])
    targets = torch.cat([torch.ones_like(positive_scores), torch.zeros_like(negative_scores)])
    return F.binary_cross_entropy_with_logits(scores, targets)


def link_metrics(embeddings, positive, negative):
    scores = torch.cat([
        dot_scores(embeddings, edge_tensor(positive)),
        dot_scores(embeddings, edge_tensor(negative)),
    ]).sigmoid().detach().numpy()
    targets = np.r_[np.ones(len(positive)), np.zeros(len(negative))]
    return roc_auc_score(targets, scores), average_precision_score(targets, scores)


seed_everything(1017)
link_model = LinkEncoder(gcn_normalize(training_adjacency))
optimizer = torch.optim.AdamW(link_model.parameters(), lr=0.02, weight_decay=1e-4)
best_validation_loss, best_link_state = float("inf"), None
for _ in range(300):
    link_model.train()
    embeddings = link_model(identity_features)
    loss = binary_edge_loss(embeddings, training_positive, training_negative)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    link_model.eval()
    with torch.no_grad():
        validation_embeddings = link_model(identity_features)
        validation_loss = binary_edge_loss(
            validation_embeddings, validation_positive, validation_negative
        ).item()
    if validation_loss < best_validation_loss:
        best_validation_loss = validation_loss
        best_link_state = copy.deepcopy(link_model.state_dict())

link_model.load_state_dict(best_link_state)
link_model.eval()
with torch.no_grad():
    link_embeddings = link_model(identity_features)
validation_auc, validation_ap = link_metrics(link_embeddings, validation_positive, validation_negative)
test_auc, test_ap = link_metrics(link_embeddings, test_positive, test_negative)

assert nx.is_connected(training_graph)
assert all(not training_graph.has_edge(*edge) for edge in validation_positive + test_positive)
print({"edge split": (len(training_positive), len(validation_positive), len(test_positive)),
       "validation AUC/AP": (round(validation_auc, 3), round(validation_ap, 3)),
       "test AUC/AP": (round(test_auc, 3), round(test_ap, 3))})
```

</details>

ROC-AUC asks whether positives tend to rank above sampled negatives; average precision is sensitive to class prevalence. Because this teaching split artificially balances positives and negatives, its AP cannot be compared with a deployed recommender containing millions of candidates. Production evaluation should mirror candidate generation, exposure, time ordering, user groups, and cold-start conditions.


### **Graph Neural Network Evaluation** {#graph-neural-network-evaluation}

Graph evaluation begins by choosing the independent unit and the information available at prediction time. A random node split on one graph tests **transductive** completion: test nodes are unlabeled during optimization but may send messages. An inductive node test removes nodes, neighborhoods, or whole graphs from training. Edge prediction must remove held-out positives from the message-passing graph. Graph classification should split entire graphs, often by scaffold, entity group, source, or time to prevent near-duplicate leakage.

Metrics follow the task. Node and edge classification use macro/micro F1, balanced accuracy, AUROC, or average precision according to imbalance and decision costs. Link ranking uses MRR, Hits@$K$, Recall@$K$, NDCG, and candidate-set-aware precision. Graph regression reports MAE/RMSE with domain-relevant units. Calibration, subgroup performance, temporal degradation, latency, memory, and sampling variance matter when predictions drive decisions.

Every learned GNN should be compared with non-neural and feature-only baselines. A graph may add no useful signal, or the labels may be predictable from degree alone. The code below compares a majority rule, logistic regression on the three structural features, and the earlier GCN on exactly the same test nodes. It also reasserts that held-out link edges never entered the encoder adjacency.

<details>
<summary><strong>Python: report baselines and verify split provenance</strong></summary>

```python
gcn_predictions = gcn_logits.argmax(1)
majority_class = int(torch.mode(labels[train_mask]).values)
majority_predictions = torch.full_like(labels[test_mask], majority_class)

feature_baseline = LogisticRegression(random_state=1018, max_iter=500)
feature_baseline.fit(node_features[train_mask].numpy(), labels[train_mask].numpy())
feature_predictions = feature_baseline.predict(node_features[test_mask].numpy())

evaluation_report = {
    "majority macro F1": f1_score(
        labels[test_mask].numpy(), majority_predictions.numpy(), average="macro"
    ),
    "feature-only macro F1": f1_score(
        labels[test_mask].numpy(), feature_predictions, average="macro"
    ),
    "GCN macro F1": f1_score(
        labels[test_mask].numpy(), gcn_predictions[test_mask].numpy(), average="macro"
    ),
    "clean link test ROC-AUC": test_auc,
    "clean link test AP": test_ap,
}

held_out_set = {tuple(sorted(edge)) for edge in validation_positive + test_positive}
training_set = {tuple(sorted(edge)) for edge in training_graph.edges()}
assert held_out_set.isdisjoint(training_set)
assert set(train_nodes).isdisjoint(test_nodes)
print({key: round(float(value), 3) for key, value in evaluation_report.items()})
```

</details>

These numbers are intentionally not elevated into a leaderboard. Nodes in one social network are correlated, the split contains only eighteen test nodes, model selection sees eight validation nodes, and one seed cannot estimate uncertainty. A credible study would pre-register multiple splits or time/scaffold partitions, tune every baseline fairly, report confidence intervals across independent graphs or seeds, and inspect performance by degree and subgroup.


### **Chapter Comparison and Summary** {#chapter-comparison-summary}

Graph neural networks replace fixed spatial or temporal coordinates with a relational neighborhood. Their central contract is simple: shared local functions process messages, invariant aggregation handles a set of neighbors, and the task head preserves the required node equivariance or graph invariance. The difficult work lies in deciding what constitutes a node, edge, relation, timestamp, negative example, and independent evaluation unit.

| Method or component | Neighborhood rule | Main advantage | Main limitation or diagnostic |
|---|---|---|---|
| Message-passing neural network | Learned messages plus invariant sum/mean/max | General framework for node, edge, and graph features | Expressiveness depends on aggregation and available graph structure |
| GCN | Fixed symmetric degree normalization | Simple, stable, and efficient sparse propagation | Transductive identity features and deep smoothing can limit transfer/depth |
| GraphSAGE | Sampled feature aggregation with self-neighbor combination | Inductive function and bounded fan-out | Sampling variance and neighborhood explosion remain |
| GAT | Learned local softmax over incident edges | Content-dependent neighbor weighting | Edge-score cost; attention weights are not causal explanations |
| Relation/temporal GNN | Type- or time-conditioned messages | Represents multiple semantics and event order | Parameter growth, temporal leakage, and sampling complexity |
| Node readout | One prediction per node state | Classification or regression on entities | Correlated node splits may overstate generalization |
| Edge readout | Pairwise endpoint/relation score | Link and relation prediction | Negative sampling and held-out-edge leakage define the result |
| Graph readout | Invariant sum/mean/max or learned pooling | One prediction per graph | Pooling may erase size or local substructure |

The Karate Club study connected every mechanism to one inspectable graph: representations remained consistent under permutation; GCN normalized degree; GraphSAGE sampled reusable features; GAT normalized attention within each target neighborhood; readouts matched output level; relation-aware code did not pretend weights were timestamps; smoothing and bottleneck diagnostics were separated; and link evaluation removed held-out edges before propagation.

The practical decision sequence is therefore:

1. define graph semantics and prediction time before selecting a layer;
2. choose node, edge, or graph output and its symmetry contract;
3. establish feature-only, topology-only, and simple propagation baselines;
4. select GCN, GraphSAGE, GAT, or typed/temporal messages according to scale and deployment;
5. inspect neighborhood growth, degree bias, oversmoothing, oversquashing, and sampling variance;
6. split by the real deployment unit and audit every feature and edge for leakage.

GNN performance is never attributable to the neural layer alone. Graph construction determines which information can travel, aggregation determines what is preserved, depth determines how far it may travel, and evaluation determines whether the reported result represents the intended future use.
